决策树由节点通过分支连接构成树形结构
因此在编写代码的时候，也要编写节点部分的代码

In [ ]:
class DecisonNode():
    def __init__(self, feature_i = None, threshold = None, value = None, true_branch=None, false_branch=None):
        feature_i = sellf.feature_i #用于分割数据的特征索引
        self.threshold = threshold    #分割特征的阈值值      
        self.value = value    #表示该叶节点的预测值  （分类树：类别标签 回归树：数值预测）        
        self.true_branch = true_branch     #满足阈值条件时进入的分支（左子树）
        self.false_branch = false_branch   #不满足阈值条件时进入左子树                                                                  

DecisionTree
决策树的基类，提供通用决策树实现框架，被回归树和分类树继承。
涉及到了继承，就要了解面向对象编程。
·进行程序设计时，主要进行数据描述以及数据处理这两个方面的工作。
·面向对象编程技术将程序设计为对象之间通过消息进行通信的相互协作。这里的对象是指具有唯一地址的、占据计算机一块内存区域的实体，由属性和行为构成。属性用数据表示，行为通过函数实现。
·抽象、封装、继承、多态是面向对象编程的基本特征。
·面向对象编程将系统分解为对象和类。class object 这个object就是类。k = object() k就是对象 对象是类的实例。

In [ ]:
class DecisionTree(object):
    """Super class of RegressionTree and ClassificationTree."""

    def __init__(self, min_samples_split=2, min_impurity=1e-7, max_depth=float("inf"), loss=None):
        
        self.root = None  
        
        self.min_samples_split = min_samples_split  #节点分裂所需的最小样本数
        
        self.min_impurity = min_impurity #分裂所需的最小增益
        
        self.max_depth = max_depth #树的最大深度 
        
        self._impurity_calculation = None #不纯度计算，未显式初始化的属性，具体实现根据树的类型
        
        self._leaf_value_calculation = None #叶节点值计算，未显式初始化的属性
       
        self.one_dim = None
        
        self.loss = loss

    def fit(self, X, y, loss=None):
        """ Build decision tree """
        self.one_dim = len(np.shape(y)) == 1 #检测one_dim是否是True，处理多输出问题的关键步骤
        self.root = self._build_tree(X, y) #调用 _build_tree() 递归构建树结构
        self.loss=None

    def _build_tree(self, X, y, current_depth=0):
        """ Recursive method which builds out the decision tree and splits X and respective y
        on the feature of X which (based on impurity) best separates the data"""

        largest_impurity = 0
        best_criteria = None    # 记录最佳分裂特征的索引和阈值
        best_sets = None        # 记录分裂后的左右子数据集
        
        # Check if expansion of y is needed
        if len(np.shape(y)) == 1:
            y = np.expand_dims(y, axis=1)

        # Add y as last column of X
        Xy = np.concatenate((X, y), axis=1)

        

        if n_samples >= self.min_samples_split and current_depth <= self.max_depth: # 检查是否满足分裂条件
            # Calculate the impurity for each feature
            for feature_i in range(n_features):
                # All values of feature_i
                feature_values = np.expand_dims(X[:, feature_i], axis=1) #从二维数组 X 中提取某一列，并扩展为二维数组
                unique_values = np.unique(feature_values) #通过提取唯一值，可以显著减少需要评估的分裂点数量。

                # Iterate through all unique values of feature column i and
                # calculate the impurity
                for threshold in unique_values: #在唯一值中选定为分裂的阈值
                    # Divide X and y depending on if the feature value of X at index feature_i
                    # meets the threshold
                    Xy1, Xy2 = divide_on_feature(Xy, feature_i, threshold)

                    if len(Xy1) > 0 and len(Xy2) > 0:
                        # Select the y-values of the two sets
                        y1 = Xy1[:, n_features:]
                        y2 = Xy2[:, n_features:]

                        # Calculate impurity
                        impurity = self._impurity_calculation(y, y1, y2)

                        # If this threshold resulted in a higher information gain than previously
                        # recorded save the threshold value and the feature
                        # index
                        if impurity > largest_impurity:
                            largest_impurity = impurity
                            best_criteria = {"feature_i": feature_i, "threshold": threshold}
                            best_sets = {
                                "leftX": Xy1[:, :n_features],   # X of left subtree
                                "lefty": Xy1[:, n_features:],   # y of left subtree 从feature_i开始到结束的y
                                "rightX": Xy2[:, :n_features],  # X of right subtree
                                "righty": Xy2[:, n_features:]   # y of right subtree
                                }

        if largest_impurity > self.min_impurity:
            # Build subtrees for the right and left branches
            true_branch = self._build_tree(best_sets["leftX"], best_sets["lefty"], current_depth + 1)
            false_branch = self._build_tree(best_sets["rightX"], best_sets["righty"], current_depth + 1)
            return DecisionNode(feature_i=best_criteria["feature_i"], threshold=best_criteria[
                                "threshold"], true_branch=true_branch, false_branch=false_branch)

        # We're at leaf => determine value
        leaf_value = self._leaf_value_calculation(y)

        return DecisionNode(value=leaf_value)


    def predict_value(self, x, tree=None):
        """ Do a recursive search down the tree and make a prediction of the data sample by the
            value of the leaf that we end up at """

        if tree is None:
            tree = self.root

        # If we have a value (i.e we're at a leaf) => return value as the prediction
        if tree.value is not None:
            return tree.value

        # Choose the feature that we will test
        feature_value = x[tree.feature_i] #输入样本 x 在当前节点的特征索引处的值

        # Determine if we will follow left or right branch
        branch = tree.false_branch
        if isinstance(feature_value, int) or isinstance(feature_value, float):
            if feature_value >= tree.threshold:
                branch = tree.true_branch
        elif feature_value == tree.threshold:
            branch = tree.true_branch

        # Test subtree
        return self.predict_value(x, branch) #根据输入的 x 找到我们应该选择的分支树结构branch，然后predict_value(x, branch)

    def predict(self, X):
        """ Classify samples one by one and return the set of labels """
        y_pred = [self.predict_value(sample) for sample in X]
        return y_pred

    def print_tree(self, tree=None, indent=" "):
        """ Recursively print the decision tree """
        if not tree:
            tree = self.root

        # If we're at leaf => print the label
        if tree.value is not None:
            print (tree.value)
        # Go deeper down the tree
        else:
            # Print test
            print ("%s:%s? " % (tree.feature_i, tree.threshold))
            # Print the true scenario
            print ("%sT->" % (indent), end="")
            self.print_tree(tree.true_branch, indent + indent)
            # Print the false scenario
            print ("%sF->" % (indent), end="")
            self.print_tree(tree.false_branch, indent + indent)

继承DecisionTree这个类，然后构建回归决策树

In [ ]:
class RegressionTree(DecisionTree):
    def _calculate_variance_reduction(self, y, y1, y2):
        var_tot = calculate_variance(y)
        var_1 = calculate_variance(y1)
        var_2 = calculate_variance(y2)
        frac_1 = len(y1) / len(y)
        frac_2 = len(y2) / len(y)

        # Calculate the variance reduction
        variance_reduction = var_tot - (frac_1 * var_1 + frac_2 * var_2)

        return sum(variance_reduction)

    def _mean_of_y(self, y):
        value = np.mean(y, axis=0)
        return value if len(value) > 1 else value[0]

    def fit(self, X, y):
        self._impurity_calculation = self._calculate_variance_reduction
        self._leaf_value_calculation = self._mean_of_y
         

继承DecisionTree这个类，然后构建分类决策树

In [ ]:
class ClassificationTree(DecisionTree):
    def _calculate_information_gain(self, y, y1, y2):
        # Calculate information gain
        p = len(y1) / len(y)
        entropy = calculate_entropy(y)
        info_gain = entropy - p * \
            calculate_entropy(y1) - (1 - p) * \
            calculate_entropy(y2)

        return info_gain

    def _majority_vote(self, y):
        most_common = None
        max_count = 0
        for label in np.unique(y):
            # Count number of occurences of samples with label
            count = len(y[y == label])
            if count > max_count:
                most_common = label
                max_count = count
        return most_common

    def fit(self, X, y):
        self._impurity_calculation = self._calculate_information_gain
        self._leaf_value_calculation = self._majority_vote
        super(ClassificationTree, self).fit(X, y)